# Lab 5 Analysis Notebook

This notebook implements TODO tasks 2-4 in a simple academic workflow:
1. Build spectral library from airborne hyperspectral data.
2. Compute false-color composites and water quality proxies.
3. Download Sentinel-2 and compute comparable indices.
4. Run minimal Spectral Angle Mapper (SAM).

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from spectral_library import build_library_from_rois, save_library_csv, save_library_metadata, load_envi_cube
from water_indices import compute_airborne_indices, compute_sentinel2_indices
from sentinel2 import search_sentinel2_l2a, download_assets
from sam import load_reference_from_library_csv, align_reference_to_cube, apply_sam_to_cube
from viewer import read_rgb

DATA_DIR = Path('data/images')
OUT_DIR = Path('data/spectral_library')
S2_DIR = Path('data/sentinel2')

hdr_files = sorted(DATA_DIR.glob('*.hdr'))
if not hdr_files:
    raise FileNotFoundError('No .hdr file found in data/images')

hdr_path = hdr_files[0]
print('Using airborne source:', hdr_path)

Using airborne source: data\images\221000_Odra_HS_Blok_A_008_VS_join_atm.hdr


In [2]:
img, wavelengths, ignore_value = load_envi_cube(hdr_path)
print(f'Cube size: {img.nrows} x {img.ncols} x {img.nbands}')
print('Ignore value:', ignore_value)

# False-color composites (examples):
# (NIR, Red, Green) and (RedEdge, Red, Green).
def closest_idx(target_nm):
    return int(np.argmin(np.abs(wavelengths - target_nm)))

fc1 = read_rgb(img, closest_idx(800), closest_idx(665), closest_idx(560), ignore_value)
fc2 = read_rgb(img, closest_idx(705), closest_idx(665), closest_idx(560), ignore_value)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(fc1)
ax[0].set_title('False color: 800/665/560 nm')
ax[0].axis('off')
ax[1].imshow(fc2)
ax[1].set_title('False color: 705/665/560 nm')
ax[1].axis('off')
plt.tight_layout()

Cube size: 4300 x 2001 x 456
Ignore value: 15000.0


## Task 2: Build spectral library

Adjust ROI coordinates below to your scene.
Each tuple is (row_min, row_max, col_min, col_max).

In [22]:
# Water ROIs built from exported viewer samples:
# spectrum_r1723_c836.csv, spectrum_r295_c191.csv, spectrum_r405_c256.csv
rois = {
    'water': [
        (1703, 1743, 816, 856),
        (275, 315, 171, 211),
        (385, 425, 236, 276),
    ],
    'green': [(220, 280, 220, 300)],
    'forest': [(320, 380, 320, 400)],
    'other': [(420, 480, 420, 500)],
}

wl, stats = build_library_from_rois(hdr_path, rois)
OUT_DIR.mkdir(parents=True, exist_ok=True)
library_csv = OUT_DIR / 'spectral_library.csv'
metadata_csv = OUT_DIR / 'spectral_library_metadata.csv'
save_library_csv(library_csv, wl, stats)
save_library_metadata(metadata_csv, hdr_path, stats)

print('Saved:', library_csv)
print('Saved:', metadata_csv)
for s in stats:
    print(f'{s.class_name}: {s.count} samples')

Saved: data\spectral_library\spectral_library.csv
Saved: data\spectral_library\spectral_library_metadata.csv
water: 4800 samples
green: 4800 samples
forest: 4800 samples
other: 4800 samples


In [14]:
# Plot spectral library means
plt.figure(figsize=(10, 5))
for s in stats:
    plt.plot(wl, s.mean, label=s.class_name)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Reflectance (DN)')
plt.title('Spectral library means')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

## Task 3: Airborne water quality proxies

In [15]:
airborne_idx = compute_airborne_indices(img, wavelengths, ignore_value=ignore_value)
print('Band mapping used:', airborne_idx['band_mapping'])

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for i, name in enumerate(['chl_a', 'doc', 'turbidity']):
    im = ax[i].imshow(airborne_idx[name], cmap='viridis')
    ax[i].set_title(f'Airborne {name}')
    ax[i].axis('off')
    plt.colorbar(im, ax=ax[i], shrink=0.7)
plt.tight_layout()

Band mapping used: {'560nm': 46, '620nm': 64, '665nm': 78, '705nm': 91}


## Task 3: Sentinel-2 download and indices

Set AOI and airborne date first.
bbox format is [min_lon, min_lat, max_lon, max_lat].

In [16]:
# Fill these values for your dataset location and date.
bbox = [16.8, 50.8, 17.0, 51.0]
airborne_date = '2022-10-01'

items = search_sentinel2_l2a(bbox=bbox, target_date=airborne_date, days_window=15, max_items=3)
if not items:
    raise RuntimeError('No Sentinel-2 items found for given AOI/date window.')

for i, item in enumerate(items):
    print(i, item.item_id, item.datetime_utc, 'cloud=', item.cloud_cover)

best = items[0]
print('Selected item:', best.item_id)
print('Assets:', sorted(best.assets.keys()))

0 S2A_33UXS_20221012_0_L2A 2022-10-12 10:06:37.750000+00:00 cloud= 1.414219
1 S2A_33UXS_20221009_0_L2A 2022-10-09 09:56:41.545000+00:00 cloud= 19.713545
2 S2B_33UXS_20221014_0_L2A 2022-10-14 09:56:34.440000+00:00 cloud= 25.670791
Selected item: S2A_33UXS_20221012_0_L2A
Assets: ['B03', 'B04', 'B05', 'B11', 'SCL']


In [17]:
# Download B03, B04, B05 to local data/sentinel2/<item_id>/
target_dir = S2_DIR / best.item_id
need = {k: v for k, v in best.assets.items() if k in {'B03', 'B04', 'B05'}}
downloaded = download_assets(need, target_dir)
downloaded

{'B03': WindowsPath('data/sentinel2/S2A_33UXS_20221012_0_L2A/B03.tif'),
 'B04': WindowsPath('data/sentinel2/S2A_33UXS_20221012_0_L2A/B04.tif'),
 'B05': WindowsPath('data/sentinel2/S2A_33UXS_20221012_0_L2A/B05.tif')}

In [18]:
import rasterio

bands = {}
for name, tif_path in downloaded.items():
    with rasterio.open(tif_path) as ds:
        bands[name] = ds.read(1).astype(np.float64)

s2_idx = compute_sentinel2_indices(bands)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for i, name in enumerate(['chl_a', 'doc', 'turbidity']):
    im = ax[i].imshow(s2_idx[name], cmap='viridis')
    ax[i].set_title(f'Sentinel-2 {name}')
    ax[i].axis('off')
    plt.colorbar(im, ax=ax[i], shrink=0.7)
plt.tight_layout()

In [23]:
# Simple global comparison (academic baseline).
for name in ['chl_a', 'doc', 'turbidity']:
    a_mean = np.nanmean(airborne_idx[name])
    s_mean = np.nanmean(s2_idx[name])
    print(f'{name}: airborne_mean={a_mean:.5f}, sentinel2_mean={s_mean:.5f}, diff={s_mean-a_mean:.5f}')

chl_a: airborne_mean=0.36636, sentinel2_mean=0.29034, diff=-0.07602
doc: airborne_mean=-0.08672, sentinel2_mean=-0.01748, diff=0.06924
turbidity: airborne_mean=0.80821, sentinel2_mean=1.00659, diff=0.19838


## Task 4: Minimal SAM and simple calibration

In [20]:
class_names = ['water', 'green', 'forest', 'other']
ref_wl, ref_matrix = load_reference_from_library_csv(library_csv, class_names)
band_idx, ref_aligned = align_reference_to_cube(wavelengths, ref_wl, ref_matrix)

# SAM on a small spatial subset for speed in notebook mode.
r0, r1 = 100, min(400, img.nrows)
c0, c1 = 100, min(400, img.ncols)
cube_subset = img.read_subregion((r0, r1), (c0, c1), bands=band_idx).astype(np.float64)
labels, min_angles = apply_sam_to_cube(cube_subset, ref_aligned)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
im0 = ax[0].imshow(labels, cmap='tab10')
ax[0].set_title('SAM class map (subset)')
ax[0].axis('off')
plt.colorbar(im0, ax=ax[0], shrink=0.8)
im1 = ax[1].imshow(min_angles, cmap='magma_r')
ax[1].set_title('SAM min angle (radians)')
ax[1].axis('off')
plt.colorbar(im1, ax=ax[1], shrink=0.8)
plt.tight_layout()

In [24]:
# Simple linear calibration: Sentinel-2 index corrected to airborne mean/variance.
# This is a minimal academic baseline, not a full physical correction.
for name in ['chl_a', 'doc', 'turbidity']:
    a = airborne_idx[name]
    s = s2_idx[name]
    a_mu, a_std = np.nanmean(a), np.nanstd(a)
    s_mu, s_std = np.nanmean(s), np.nanstd(s)
    s_cal = (s - s_mu) * (a_std / (s_std + 1e-8)) + a_mu
    print(f'{name}: mean_before={s_mu:.5f}, mean_after={np.nanmean(s_cal):.5f}, target_airborne={a_mu:.5f}')

chl_a: mean_before=0.29034, mean_after=0.36636, target_airborne=0.36636
doc: mean_before=-0.01748, mean_after=-0.08672, target_airborne=-0.08672
turbidity: mean_before=1.00659, mean_after=0.80821, target_airborne=0.80821


## Extra plots for report

The two plots below summarize index comparability and SAM confidence.

In [27]:
# Bar plot: mean values of indices before and after calibration.
index_names = ['chl_a', 'doc', 'turbidity']
a_means = [np.nanmean(airborne_idx[n]) for n in index_names]
s_means = [np.nanmean(s2_idx[n]) for n in index_names]
s_cal_means = []
for n in index_names:
    a = airborne_idx[n]
    s = s2_idx[n]
    a_mu, a_std = np.nanmean(a), np.nanstd(a)
    s_mu, s_std = np.nanmean(s), np.nanstd(s)
    s_cal = (s - s_mu) * (a_std / (s_std + 1e-8)) + a_mu
    s_cal_means.append(np.nanmean(s_cal))

x = np.arange(len(index_names))
width = 0.25

plt.figure(figsize=(9, 4.8))
plt.bar(x - width, a_means, width=width, label='Airborne mean')
plt.bar(x, s_means, width=width, label='Sentinel-2 mean')
plt.bar(x + width, s_cal_means, width=width, label='Sentinel-2 calibrated mean')
plt.xticks(x, index_names)
plt.ylabel('Mean index value')
plt.title('Index means: airborne vs Sentinel-2 vs calibrated')
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()

In [28]:
# Histogram of SAM minimum angles as a confidence proxy.
valid_angles = min_angles[np.isfinite(min_angles)]

plt.figure(figsize=(8, 4.5))
plt.hist(valid_angles.ravel(), bins=40, color='steelblue', edgecolor='white', alpha=0.9)
plt.xlabel('SAM minimum angle (radians)')
plt.ylabel('Pixel count')
plt.title('Distribution of SAM minimum angles')
plt.grid(alpha=0.25)
plt.tight_layout()